# Lexos SeeTrees Tutorial

The `seetrees` module provides stylometric analysis and visualization tools for document-term-style data. It is especially useful for comparing document profiles, computing distance matrices, visualizing document relationships, and exploring feature importance across clusters.

It can compute stylometric distance matrices using several metrics, visualize relationships using Multidimensional Scaling (MDS) or Principal Components Analysis (PCA), compare documents by z-score profiles, and render dendrograms with feature importance scores.

SeeTrees is adapted from the R ['see' package](https://github.com/perechen/seetrees){target="_blank"} by Artjoms Šeļa.

The module's `SeeTrees` class accepts frequency tables and distance matrices as input, but the recommended method is to use a Lexos `DTM` object. So we'll begin by importing some data and creating a DTM.

In [ ]:
from lexos import DTM
from lexos import Loader
from lexos import Tokenizer
from lexos.scrubber.normalize import whitespace

# Load some text files and set their names
files = [
    "FilesToUse/Poe_FallOfHouseUsher_1839.txt",
    "FilesToUse/Lippard_BelOfPrairieEden.txt",
    "FilesToUse/Irving_RipVanWInkle.txt",
    "FilesToUse/HenryWP_ThePirate.txt",
]
loader = Loader()
loader.load(files)
loader.names = [
    "Poe_Usher",
    "Lippard_Bel",
    "Irving_RipVW",
    "Henry_Pirate"
    ]

# Scrub extra newlines and whitespace from the loaded documents
loader.texts = [text.replace("\n", " ") for text in loader.texts]
loader.texts = [whitespace(text) for text in loader.texts]

# Tokenize the loaded documents
tokenizer = Tokenizer()
docs = list(tokenizer.make_docs(texts=loader.texts))
labels = loader.names

print(f"Loaded {len(docs)} documents with labels: {labels}")

# Create a Document-Term Matrix (DTM)
dtm = DTM()
dtm(docs=docs, labels=labels)

print(f"DTM created with {dtm.to_df().shape[1]} documents and {dtm.to_df().shape[0]} unique terms.")

### Creating a SeeTrees Instance 

Once the `SeeTrees` instance is created with the DTM, the `get_tree` method produces two plots: a dendrogram cut to groups and lists of words associated with those groups.

Words associated with clusters are determined by calculating correlation ratio η^2^ of word frequency (f) across clusters (c) and documents (d). Then results are filtered by p-value (which might not make sense at all). See the [User Guide](https://scottkleinman.github.io/lexos/dev/user_guide/cluster/seetrees/#statistical-concepts-used-by-seetrees) for an explanation of the statistical concepts.

In [ ]:
# Import the SeeTrees class
from lexos.cluster.seetrees import SeeTrees

# Create an instance of the SeeTrees object (feel free to change the parameters)
st = SeeTrees(dtm=dtm)

## Computing Stylometric Distances

To compare how "distant" authors or texts are from one another, you must compute a distance matrix with the SeeTrees `compute_distances` method.

SeeTrees supports five common metrics for producing pairwise distance matrices from frequency data:

- `manhattan`: Sum of the absolute differences between the raw frequencies of every term the documents.
- `cosine`: Cosine distance calculated using raw frequencies to measure the angle between document vectors in a multi-dimensional space.
- `delta` (for Burrows' Delta): Manhattan distance over z-scores.
- `eder_delta` (for Eder's Delta): Rank-weighted Manhattan distance calculated over z-scores.
- `cosine_delta`: Cosine distance calculated using z-score frequencies.

For further details on these metrics and how to choose them, see the [SeeTrees page](https://scottkleinman.github.io/lexos/dev/user_guide/cluster/seetrees/#statistical-concepts-used-by-seetrees){target="_blank"} in the Lexos User Guide.

You can supply your chosen metric with the `metric` parameter:

In [ ]:
# Compute distances using Burrows' Delta
distance_matrix = st.compute_distances(metric="delta")
distance_matrix.head()

## Analyzing Feature Importance (Z-Scores)

SeeTrees uses z-scores to measure how many standard deviations a feature's frequency is from the mean across the corpus. This allows you to identify which features are most distinctive for a specific author or text.

- A positive z-score indicates that a feature (e.g., a specific word) is used **more frequently** in that document than the average usage across the entire corpus.
- A negative z-score indicates that the feature is used **less frequently** than the corpus average.
- A near zero z-score indicates the feature's usage is close to the average for the corpus.

### Viewing Distinctive Features for One Text

To see the top most distinctive terms for a target text, use `get_feature_score_plot` for a chart and `get_feature_summary` for a DataFrame.

`get_feature_score_plot` returns a Plotly-based chart object. The bar chart displays preferred (pink) and avoided (light blue) words. Dashed lines mark the mean, +-1 and +-2 standard deviations.

The plot can get crowded, so you may have to use the pan and zoom function from the toolbar that appears when you hover over the plot. In many cases, the top features have similar z-scores, and you won't start to see variation until far down the list. That is why the example below is set to show the top 250 features.

**Note:** If you haven't scrubbed linebreaks and whitespace from your texts, they will appear as invisible textual features in the z-scores. In order to make them visible, they are converted to "<linebreak>" and "<whitespace>".


In [ ]:
st.get_feature_score_plot(target_text="Henry_Pirate", top=250).show()

If the plot does not provide readable or usable results, you can also obtain the z-scores as a pandas DataFrame.

In [ ]:
# Tabular summary
st.get_feature_summary(target_text="Henry_Pirate", top=10)

### Comparing Two Documents

The `get_overlay_plot` and `get_difference_plot` methods allow you to interpret document relationships by visualizing them in two different ways:

- **Overlay Profile**: This view overlays the z-score profiles of two different documents. By looking at where the peaks and valleys align or diverge, you can see if two authors share similar stylistic habits (e.g., both using specific function words at a high rate) or if their styles are distinct.
- **Difference Profile**: This displays a bar chart of the direct z-score differences between a target and a source document. It helps you pinpoint exactly which words are responsible for the stylistic distance between two texts.

When using either view, you can control the number of top features displayed by using the `top_diff` parameter.

For further details on how these profiles are calculated, see the [SeeTrees page](https://scottkleinman.github.io/lexos/dev/user_guide/cluster/seetrees/#statistical-concepts-used-by-seetrees){target="_blank"} in the Lexos User Guide.

In [ ]:
# Compare two authors' profiles
fig = st.get_overlay_plot(
    source_text="Poe_Usher",
    target_text="Henry_Pirate",
    top_diff=10,
    max_rank=100,
 )
fig.show()

In [ ]:
# Compare two authors' differences
fig = st.get_difference_plot(source_text="Poe_Usher",
                             target_text="Henry_Pirate")
fig.show()

## Visualizing Document Relationships

You can understand the "stylistic space" of your corpus using Principal Components Analysis (PCA), Multidimensional Scaling (MDS), and density plots. These methods visualize document relationships in two-dimensional space.

Note that `get_mds_plot(...)` and `get_density_plot(...)` require a precomputed distance table. Run `compute_distances(...)` first.

### Principal Components Analysis

The PCA method focuses on the **variance in feature usage**. It projects the frequency-table data into two dimensions so you can inspect stylistic proximity.

In [ ]:
# Visualize relationships using PCA
fig = st.get_pca_plot()
fig.show()

### Multidimensional Scaling

The MDS method focuses on maintaining the relative distances between documents as defined by a stylometric metric. MDS requires a distance matrix to be available, so compute it first with `compute_distances(metric=...)`.

In [ ]:
# Visualize relationships using MDS
st.compute_distances(metric="delta")
fig = st.get_mds_plot()
fig.show()

### Density Plots

You can use a density plot to look at how similar or different pairs of texts are in aggregate.

- Each pair of texts gets a distance score.
- The plot draws a smooth curve showing how common each distance value is.
- If two texts are very similar, their distance is small; if they are different, the distance is larger.

This is useful when you want to see a broad view of how text pairs are distributed by distance, to compare same-author vs different-author similarity, or to highlight one author’s distances. It is a higher-level, aggregate way to understand stylistic distance, rather than plotting individual documents in 2D as we do with PCA and MDS.

To draw one curve for pairs from the same author/class and another curve for pairs from different authors/classes, set `group=True` (the default). To draw a single curve for all pairs, set `group=False`.

To highlight one author/class inside the density plot, set the `author` parameter. This is how the function decides whether two documents belong to the class. By default, the function looks for the value you set in document labels, separated by an underscore. For instance, if you have labels like "Austen_Emma" and "Austen_Pride", and you set `author=Austen`, that will be identified as the class name and all documents beginning with "Austen_" will be grouped within this class. If the default behaviour does not work for you, it is possible to customise the method of parsing labels by providing your own regex pattern to the `pattern` parameter.

Internally, density view reads the list of document labels from `distance_table` and assigns each label to a class. It then computes all pairwise distances from the lower half of the distance matrix and for each pair records the distance value, whether they belong to the same class, and the class label of the first document in the pair. It then makes a table of these values and plots density curves based on the values in the table.

If you set `group=True`, you get two curves: one for same-author pairs and one for different-author pairs. If the same-author curve is far to the left, that means same-author texts are generally closer together. If the different-author curve is farther right, that means different-author texts are more distant.

If `group=False`, you will see a single overall distance distribution. This is useful to understand the general range of distances in the dataset.

In [ ]:
# Visualize relationships with a density plot
fig = st.get_density_plot(group=True, author="Poe", pattern=r"^.*?(?=_)")
fig.show()

## Hierarchical Clustering and Dendrograms

The `get_tree()` method implements hierarchical agglomerative clustering and renders a dendrogram to show how documents cluster together stylistically. In this respect, it is like the Lexos `dendrogram` module. However, SeeTrees offers additional functionality to help you understand the textual features responsible for the clustering.

When you cut the tree into (`k`) clusters, SeeTrees calculates **Eta-squared** (*η*^2^), which represents the proportion of total variance in a feature explained by cluster assignment. This identifies the "top terms" that define the clusters. In other words, while a dendrogram shows *that* texts are similar, eta-squared explains *why* they were grouped that way by pointing to the most influential terms.

Use the `label_buffer` parameter to add space on the active label side of the dendrogram.

**Notes:**

- If you encounter a `ValueError: Frequency data is required`, ensure you initialized SeeTrees with a valid frequency table.
- Always run `compute_distances()` before calling `get_tree()` to avoid errors.


In [ ]:
# Create a dendrogram with 3 clusters
# Ensure distances are computed first
st.compute_distances(metric="cosine_delta")
st.get_tree(k=3, orientation="left").show()

## Saving Plots

Most of the plots produced in the `seetrees` module are Matplotlib-based objects. To save them, capture the underlying figure and call `savefig`:

```python
tree_plot = st.get_tree(k=3)
tree_plot.fig.savefig("output.png")
```

Changing the extension to `.pdf` would save the file in that format. For other available parameters, see the [`matplotlib` documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html).

The output of `get_feature_score_plot` is a Plotly-based object. You can save the underlying Plotly figure with `write_image`:

```python
score_plot = st.get_feature_score_plot(target_text="Henry_Pirate", top=10)
plotly_fig = score_plot.plot()
plotly_fig.write_image("output.png")
```

For available parameters, see the [Plotly documentation](https://plotly.github.io/plotly.py-docs/generated/plotly.io.write_image.html).

If you want to save the interactive plot, use `write_html`:

```python
plotly_fig.write_html("my_chart.html")
```

You can reduce saved file size by loading `plotly.js` from a CDN:

```python
plotly_fig.write_html("my_chart.html", include_plotlyjs="cdn")
```

For available parameters in `write_html`, see the [Plotly documentation](https://plotly.github.io/plotly.py-docs/generated/plotly.io.write_html.html).
